# RIFT-SVC 25K inference - Float WAV

This notebook runs the inference-only RIFT-SVC 25K checkpoint on one explicitly selected vocal stem. It writes a 44.1 kHz mono Float WAV, suitable for further mixing or editing.

Enable Internet and GPU, then add the vocal input as a Kaggle Dataset.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'RIFT-SVC'
ASSET_DIR = WORKDIR / 'rift-assets'
OUTPUT_DIR = WORKDIR / 'outputs'
MODEL_REPO = 'ooaaqq/rift-svc-luzao-25k'
MODEL_SHA256 = '3db77a14098d87359dd69156973e2e315c642da8df17de1686831c91faed0c86'

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    pass

if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'master',
        'https://github.com/ooaaqq/RIFT-SVC.git', str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements-kaggle.txt')
], check=True)
print('Repository:', subprocess.check_output([
    'git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'
], text=True).strip())

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not available. Enable a GPU accelerator.')
print('GPU:', torch.cuda.get_device_name(0))

# Select the Stage 2 BS-Roformer 124 bands vocal stem explicitly.
# Change only this filename when running another prepared stem.
INPUT_AUDIO_NAME = (
    '20260812011254-b8a9f604c1-no-4-self_'
    'bs_roformer_mt_171_vocals_[mvsep.com].wav'
)
# Kaggle may strip square brackets from uploaded filenames.
INPUT_AUDIO_ALIASES = {
    INPUT_AUDIO_NAME,
    INPUT_AUDIO_NAME.replace('[', '').replace(']', ''),
}
audio_suffixes = {'.wav', '.flac', '.mp3', '.m4a', '.ogg', '.opus'}
all_audio = sorted(
    path for path in Path('/kaggle/input').rglob('*')
    if path.is_file() and path.suffix.lower() in audio_suffixes
)
matches = [path for path in all_audio if path.name in INPUT_AUDIO_ALIASES]
if len(matches) != 1:
    available = '\n'.join(str(path) for path in all_audio)
    raise RuntimeError(
        f'Expected exactly one input named {INPUT_AUDIO_NAME!r}, found {len(matches)}.\n'
        f'Available audio files:\n{available}'
    )
input_audio = matches[0]
print('Input:', input_audio)
print('All attached audio files:', len(all_audio))

In [ ]:
ASSET_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, str(REPO_DIR / 'scripts/download_inference_assets.py'),
    '--model-repo', MODEL_REPO,
    '--model-filename', 'rift25k.ckpt',
    '--output-dir', str(ASSET_DIR),
    '--expected-sha256', MODEL_SHA256,
], check=True)

In [ ]:
# Edit this block for a new candidate. The output name follows these values.
SPEAKER = 'target'
KEY_SHIFT = 0
DEVICE = 'cuda'
INFER_STEPS = 32
DS_CFG_STRENGTH = 0.2
SPK_CFG_STRENGTH = 0.8
CFG_RESCALE = 0.7
ROBUST_F0 = 0
SEED = 7

def format_name_value(value):
    return str(value).replace('-', 'm').replace('.', 'p')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
name_suffix = (
    f'rift25k-spk-{SPEAKER}-k{format_name_value(KEY_SHIFT)}'
    f'-steps{INFER_STEPS}'
    f'-ds{format_name_value(DS_CFG_STRENGTH)}'
    f'-spk{format_name_value(SPK_CFG_STRENGTH)}'
    f'-cfg{format_name_value(CFG_RESCALE)}'
    f'-rf{ROBUST_F0}-seed{SEED}'
)
output_audio = OUTPUT_DIR / f'{input_audio.stem}__{name_suffix}.wav'
inference_command = [
    sys.executable, str(REPO_DIR / 'infer.py'),
    '--model', str(ASSET_DIR / 'model' / 'rift25k.ckpt'),
    '--assets-dir', str(ASSET_DIR / 'pretrained'),
    '--input', str(input_audio),
    '--output', str(output_audio),
    '--speaker', SPEAKER,
    '--key-shift', str(KEY_SHIFT),
    '--device', DEVICE,
    '--infer-steps', str(INFER_STEPS),
    '--ds-cfg-strength', str(DS_CFG_STRENGTH),
    '--spk-cfg-strength', str(SPK_CFG_STRENGTH),
    '--cfg-rescale', str(CFG_RESCALE),
    '--robust-f0', str(ROBUST_F0),
    '--output-subtype', 'FLOAT',
    '--seed', str(SEED),
]
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
print('Command:', ' '.join(inference_command))
print('Starting inference subprocess...', flush=True)
process = subprocess.Popen(
    inference_command,
    cwd=REPO_DIR,
    env=run_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'RIFT inference failed with exit code {return_code}. '
        'The complete child-process traceback is shown above.'
    )
print('Output:', output_audio)
print('Size:', output_audio.stat().st_size, 'bytes')

In [ ]:
import soundfile as sf
info = sf.info(output_audio)
print('Sample rate:', info.samplerate)
print('Channels:', info.channels)
print('Duration:', info.duration)
print('Subtype:', info.subtype)
assert info.format == 'WAV'
assert info.subtype == 'FLOAT'

In [ ]:
import shutil

archive = shutil.make_archive(
    '/kaggle/working/rift25k-inference-wav',
    'zip',
    root_dir=OUTPUT_DIR,
)
print('Download archive:', archive)

from IPython.display import FileLink
FileLink('rift25k-inference-wav.zip')

### Temporary external download link (optional)

The next cell uploads the ZIP to a temporary file service and prints a public download URL. The link expires according to the selected service policy.

In [ ]:
import json

archive_path = Path('/kaggle/working/rift25k-inference-wav.zip')
if not archive_path.is_file() or archive_path.stat().st_size == 0:
    raise FileNotFoundError(f'Archive is missing or empty: {archive_path}')

def parse_upload_response(service, response):
    lines = [line.strip() for line in response.splitlines() if line.strip()]
    if lines and lines[-1].startswith(('http://', 'https://')):
        return lines[-1]
    try:
        payload = json.loads(response)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'{service} returned an unexpected response: {response!r}') from exc

    if service == 'tmpfiles.org':
        url = payload.get('data', {}).get('url')
        if url and '://tmpfiles.org/' in url:
            return url.replace('://tmpfiles.org/', '://tmpfiles.org/dl/', 1)
    for key in ('link', 'directLink', 'url'):
        url = payload.get(key)
        if isinstance(url, str) and url.startswith(('http://', 'https://')):
            return url
    raise RuntimeError(f'{service} returned no download URL: {response!r}')

def run_upload(service, command):
    result = subprocess.run(command, capture_output=True, text=True, timeout=1800)
    if result.returncode != 0:
        detail = (result.stderr or result.stdout).strip()
        raise RuntimeError(detail or f'upload exited with {result.returncode}')
    return parse_upload_response(service, result.stdout.strip())

curl_options = [
    'curl', '--fail', '--silent', '--show-error', '--retry', '2',
    '--retry-delay', '3', '--connect-timeout', '20', '--max-time', '1800',
]
uploaders = [
    ('litterbox.catbox.moe', curl_options + [
        '-F', 'reqtype=fileupload', '-F', 'time=72h',
        '-F', f'fileToUpload=@{archive_path}',
        'https://litterbox.catbox.moe/resources/internals/api.php',
    ]),
    ('catbox.moe', curl_options + [
        '-F', 'reqtype=fileupload', '-F', f'fileToUpload=@{archive_path}',
        'https://catbox.moe/user/api.php',
    ]),
    ('tmpfiles.org', curl_options + [
        '-F', f'file=@{archive_path}', 'https://tmpfiles.org/api/v1/upload',
    ]),
    ('file.io', curl_options + [
        '-F', f'file=@{archive_path}', 'https://file.io',
    ]),
    ('0x0.st', curl_options + [
        '-F', f'file=@{archive_path}', 'https://0x0.st',
    ]),
    ('transfer.sh', curl_options + [
        '--upload-file', str(archive_path),
        f'https://transfer.sh/{archive_path.name}',
    ]),
]

for service, command in uploaders:
    try:
        temporary_url = run_upload(service, command)
        print(f'{service} temporary download URL (expires according to service policy):')
        print(temporary_url)
        break
    except Exception as exc:
        print(f'{service} upload failed: {exc}')
else:
    print('All temporary upload services failed; use the FileLink above or retry later.')